**Kaggle:** Settings -> Accelerator **GPU T4 x2**, Internet **On** (phone-verified account), then **Save Version -> Save & Run All (Commit)**; the run continues after you close the tab (about 7-8 h on a T4, under Kaggle's 12 h limit). When it finishes, download `cad-distill-<variant>-results.zip` from the version's **Output** tab.

# CAD editing: verifier-guided distillation, Astra teacher -> Qwen2.5-Coder-3B student

Same QLoRA recipe, splits and scorer as the completed 3B run; only the training targets change.
Targets are `PLAN:` (quoted zero-based original lines) then `PATCH:` (the JSON patch).

* **template**: plans generated from the reference patch, no teacher (control; isolates the plan format).
* **distill**: BenchCAD training rows whose Astra candidates pass the target-free CAD verifier (execution gates + numeric checks + geometric consensus of n samples) use the teacher's answer.
* **distill-ref**: as distill, keeping only teacher answers that also match the training reference.

Use **Runtime -> Change runtime type -> GPU** (A100/L4 is ~4x faster than T4), then **Run all**. No API keys or passwords are in this notebook; teacher outputs were generated beforehand on the training split only.

In [ ]:
#@title Kaggle: keep everything under /kaggle/working (the bundle's /content paths point here)
import os
# One GPU only: the recipe is single-GPU (effective batch 4). With T4 x2 visible, the HF Trainer wraps the 4-bit
# model in DataParallel (effective batch 8, 634 steps) and bitsandbytes crashes with an illegal memory access.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
work = Path('/kaggle/working/cad_distill'); work.mkdir(parents=True, exist_ok=True)
Path('/content').mkdir(exist_ok=True)
link = Path('/content/cad_distill')
if not link.exists(): link.symlink_to(work)
print('outputs ->', link.resolve())

In [ ]:
!pip -q install "transformers==4.51.3" "peft==0.15.2" accelerate bitsandbytes "cadquery==2.8.0"
!apt-get -qq update && apt-get -qq install -y libglu1-mesa libgl1
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(torch.cuda.get_device_name(0), torch.cuda.mem_get_info())

In [ ]:
#@title Unpack data, scripts and test reference STEP files (from the attached Kaggle dataset)
import hashlib, io, json, tarfile
from pathlib import Path
source = next(Path('/kaggle/input').rglob('cad-distill-bundle.bin'))
bundle = source.read_bytes()
assert hashlib.sha256(bundle).hexdigest() == "d4c9022af9003f32febf6d0b4bb032de8917316a088b184d7143387712440762", 'bundle differs from the built notebook'
ROOT = Path('/content/cad_distill'); ROOT.mkdir(exist_ok=True)
tarfile.open(fileobj=io.BytesIO(bundle), mode='r:gz').extractall(ROOT)
for v in ['template', 'distill', 'distill-ref']:
    m = json.loads((ROOT/'data'/v/'manifest.json').read_text())
    print(v, m['counts']['train'], m.get('acceptance'), m.get('teacher_audit'))

In [ ]:
#@title Which student to train
VARIANTS = ['distill']  #@param {type:"raw"}
# distill: verifier-selected teacher answers (consensus of 2 Astra samples), no reference used
# distill-ref: additionally keeps only teacher answers whose solid matches the reference (training split)
# Each takes ~1.3 h on an L4; drop one to halve the run. 'template' results already exist.

## Train, evaluate and decode validation (background job)

Two epochs; final checkpoint; greedy decoding on all 252 held-out tasks, then on the 172 validation tasks (used offline to check the plan-quote anchoring rule on data that played no part in designing it). Runs as a background process logging to `train.log`, so a dropped VS Code or browser connection does not stop it: reconnect and re-run the wait cell. A completed variant is never retrained; an incomplete one is moved aside and restarted.

In [ ]:
import json, os, subprocess, sys, time
LOG, PID = ROOT/'train.log', ROOT/'train.pid'
def status(v):  # completed only if trained on exactly the current data files
    m = ROOT/f'results-{v}'/'run_manifest.json'
    if not m.exists(): return None
    run = json.loads(m.read_text())
    same = run.get('source_manifest', {}).get('files') == json.loads((ROOT/'data'/v/'manifest.json').read_text())['files']
    return run['status'] if same else 'stale'
def alive():
    if LOG.exists() and 'EXIT=' in LOG.read_text(errors='replace')[-300:]: return False
    try:
        os.kill(int(PID.read_text()), 0); return True
    except (OSError, ValueError):
        return False
steps = []
for v in VARIANTS:
    run = ROOT/f'results-{v}'
    if status(v) != 'completed':
        if run.exists() and not alive(): run.rename(run.with_name(f'{run.name}.incomplete-{int(time.time())}'))
        steps.append(f'{sys.executable} -u {ROOT}/scripts/train_distill_cad_colab.py --data {ROOT}/data/{v} --output {run}')
    if not (run/'validation-predictions.jsonl').exists():
        steps.append(f'{sys.executable} -u {ROOT}/scripts/decode_split_colab.py --data {ROOT}/data/{v} --run {run} --split validation')
if alive():
    print('Job already running: go to the next cell')
elif not steps:
    print('All requested variants are trained and decoded')
else:
    proc = subprocess.Popen(f'(cd {ROOT}/scripts && ' + ' && '.join(steps) + f') > {LOG} 2>&1; echo EXIT=$? >> {LOG}', shell=True, start_new_session=True)
    PID.write_text(str(proc.pid)); print('Started', len(steps), 'steps, pid', proc.pid)

In [ ]:
#@title Wait for training (re-run after a reconnect)
seen = 0
while True:
    lines = LOG.read_text(errors='replace').replace('\r', '\n').splitlines() if LOG.exists() else []
    for line in lines[seen:]:
        if any(k in line for k in ("'loss'", "'eval_loss'", 'trained ', 'validation ', 'EXIT=', 'Error', 'Traceback', 'train_runtime')):
            print(line[:300], flush=True)
    seen = len(lines)
    if any(l.startswith('EXIT=') for l in lines): break
    time.sleep(60)
assert 'EXIT=0' in lines[-1], '\n'.join(lines[-60:])
print('Training, test and validation decoding finished')

## Execute predictions against the reference solids

BenchCAD uses author-released STEP targets; CAD-Editor uses locally reconstructed sequence targets. Strict = volume IoU >= 0.99999.

In [ ]:
for v in VARIANTS:
    run = ROOT/f'results-{v}'
    subprocess.run([sys.executable, '-u', str(ROOT/'scripts/score_multisource_cad_geometry.py'), str(run/'trained-predictions.jsonl')], check=True, cwd=ROOT/'scripts', stdout=subprocess.DEVNULL)
baselines = {"3B SFT, patch only (runs/multisource-cad-colab-20261002, local re-score)": [39, 25, 14], "Astra gpt-6-astra, zero-shot (runs/multisource-cad-astra-20261004)": [106, 89, 17]}
print(f"{'system':78s} strict  BenchCAD  CAD-Editor")
for name, (a, b, c) in baselines.items(): print(f'{name:78s} {a:6d} {b:9d} {c:11d}')
for v in VARIANTS:
    g = json.loads((ROOT/f'results-{v}'/'trained-predictions-geometry.json').read_text())
    print(f"{'3B student, '+v:78s} {g['match_strict']:6d} {g['sources']['BenchCAD']['match_strict']:9d} {g['sources']['CAD-Editor']['match_strict']:11d}")

## Save results

Writes `cad-distill-<variant>-results.zip` (everything except training checkpoints, adapter weights included) to /kaggle/working for the Output tab.

In [ ]:
import zipfile
for v in VARIANTS:
    run = ROOT/f'results-{v}'
    archive = Path(f'/kaggle/working/cad-distill-{v}-results.zip')
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in run.rglob('*'):
            if f.is_file() and 'checkpoints' not in f.parts: z.write(f, f.relative_to(run))
    print('Saved', archive, archive.stat().st_size, 'bytes')